# 03_Dataset_Merging_and_Preparation.ipynb

This notebook focuses on merging the extracted landmark data and preparing it for model training. It will combine individual CSV files containing landmark data for different gestures into a single, unified dataset.

In [3]:
import pandas as pd
import os
import numpy as np

print("Libraries imported successfully.")

Libraries imported successfully.


## Configuration

Define paths for raw landmark data and the output merged dataset.

In [4]:
RAW_LANDMARKS_DIR = '../model_artifacts/raw_landmarks'
OUTPUT_DIR = '../model_artifacts'
OUTPUT_FILENAME = 'keypoint.csv'
OUTPUT_LABEL_FILENAME = 'keypoint_classifier_label.csv'

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Raw landmarks directory: {RAW_LANDMARKS_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

Raw landmarks directory: ../model_artifacts/raw_landmarks
Output directory: ../model_artifacts


## Load and Merge Landmark Data

This section iterates through the raw landmark CSV files, assigns a label to each gesture, and merges them into a single DataFrame.

In [5]:
def merge_landmark_data(raw_landmarks_dir):
    all_data = []
    labels = []
    gesture_map = {}
    label_id = 0

    for filename in os.listdir(raw_landmarks_dir):
        if filename.endswith('.csv'):
            gesture_name = os.path.splitext(filename)[0]
            file_path = os.path.join(raw_landmarks_dir, filename)
            
            if gesture_name not in gesture_map:
                gesture_map[gesture_name] = label_id
                labels.append([gesture_name, label_id])
                label_id += 1
            
            current_label_id = gesture_map[gesture_name]
            
            df = pd.read_csv(file_path, header=None)
            df['label'] = current_label_id
            all_data.append(df)
            print(f"Loaded {filename} with label {current_label_id}")

    if not all_data:
        print("No CSV files found in the raw landmarks directory.")
        return None, None

    merged_df = pd.concat(all_data, ignore_index=True)
    labels_df = pd.DataFrame(labels, columns=['gesture', 'label_id'])
    
    return merged_df, labels_df

merged_df, labels_df = merge_landmark_data(RAW_LANDMARKS_DIR)

if merged_df is not None:
    print(f"\nMerged DataFrame shape: {merged_df.shape}")
    print("Merged DataFrame head:")
    print(merged_df.head())
    print("\nLabels DataFrame:")
    print(labels_df)
else:
    print("Failed to merge data.")

Loaded afraid.csv with label 0
Loaded agree.csv with label 1
Loaded assistance.csv with label 2
Loaded bad.csv with label 3
Loaded become.csv with label 4
Loaded college.csv with label 5
Loaded doctor.csv with label 6
Loaded from.csv with label 7
Loaded pain.csv with label 8
Loaded pray.csv with label 9
Loaded secondary.csv with label 10
Loaded skin.csv with label 11
Loaded small.csv with label 12
Loaded specific.csv with label 13
Loaded stand.csv with label 14
Loaded today.csv with label 15
Loaded warn.csv with label 16
Loaded which.csv with label 17
Loaded work.csv with label 18
Loaded you.csv with label 19

Merged DataFrame shape: (14906, 44)
Merged DataFrame head:
       0          1          2                     3                     4  \
0  class  feature_0  feature_1             feature_2             feature_3   
1      0        0.0        0.0  -0.16666666666666666                  -0.2   
2      0        0.0        0.0   0.13186813186813187   -0.0989010989010989   
3      0   

## Save Merged Data and Labels

The merged landmark data and the corresponding labels are saved to CSV files in the `model_artifacts` directory.

In [6]:
if merged_df is not None and labels_df is not None:
    output_keypoint_path = os.path.join(OUTPUT_DIR, OUTPUT_FILENAME)
    output_label_path = os.path.join(OUTPUT_DIR, OUTPUT_LABEL_FILENAME)
    
    merged_df.to_csv(output_keypoint_path, index=False, header=False)
    labels_df.to_csv(output_label_path, index=False, header=False)
    
    print(f"\nMerged keypoint data saved to: {output_keypoint_path}")
    print(f"Labels saved to: {output_label_path}")
else:
    print("Skipping save operation as no data was merged.")


Merged keypoint data saved to: ../model_artifacts\keypoint.csv
Labels saved to: ../model_artifacts\keypoint_classifier_label.csv


## Data Verification

Quick check to ensure the saved files can be loaded correctly.

In [7]:
try:
    loaded_keypoints = pd.read_csv(os.path.join(OUTPUT_DIR, OUTPUT_FILENAME), header=None)
    loaded_labels = pd.read_csv(os.path.join(OUTPUT_DIR, OUTPUT_LABEL_FILENAME), header=None)
    
    print(f"\nSuccessfully loaded keypoint data. Shape: {loaded_keypoints.shape}")
    print(f"Successfully loaded labels data. Shape: {loaded_labels.shape}")
    print("First 5 rows of loaded keypoints:")
    print(loaded_keypoints.head())
    print("First 5 rows of loaded labels:")
    print(loaded_labels.head())
except Exception as e:
    print(f"Error during data verification: {e}")


Successfully loaded keypoint data. Shape: (14906, 44)
Successfully loaded labels data. Shape: (20, 2)
First 5 rows of loaded keypoints:
      0          1          2                     3                     4   \
0  class  feature_0  feature_1             feature_2             feature_3   
1      0        0.0        0.0  -0.16666666666666666                  -0.2   
2      0        0.0        0.0   0.13186813186813187   -0.0989010989010989   
3      0        0.0        0.0   -0.3157894736842105   -0.1368421052631579   
4      0        0.0        0.0  -0.22115384615384615  0.009615384615384616   

                     5                     6                    7   \
0             feature_4             feature_5            feature_6   
1  -0.19166666666666668   -0.4083333333333333                -0.25   
2   0.26373626373626374  -0.25274725274725274  0.25274725274725274   
3  -0.47368421052631576  -0.35789473684210527   -0.631578947368421   
4   -0.4230769230769231  -0.0769230769230769